# 第五课：类、实例与面向对象

本课目标：

- 理解类与实例的关系；
- 理解`__init__`和`self`；
- 区分实例属性与类属性；
- 编写实例方法；
- 判断什么时候值得使用类。

本课作业会实现一个小型`ChatSession`，它以后可以发展成AI聊天系统中的会话对象。

## 1. 类与实例

类可以理解为创建对象的规则，实例是按照该规则创建的具体对象：

```python
class User:
    pass

alice = User()
bob = User()
```

`alice`和`bob`属于同一个类，但它们是两个不同实例。

In [ ]:
class User:
    pass


alice = User()
bob = User()

print(type(alice))
print(alice is bob)
print(isinstance(alice, User))

## 2. `__init__`与`self`

创建实例时，Python会调用`__init__()`进行初始化：

```python
class User:
    def __init__(self, name: str):
        self.name = name
```

`self`表示当前正在操作的实例。`self.name`是保存在该实例上的属性。调用时不需要手动传入`self`。

In [ ]:
class User:
    def __init__(self, name: str):
        self.name = name

    def introduce(self) -> str:
        return f"我是{self.name}"


alice = User("Alice")
bob = User("Bob")
print(alice.name)
print(bob.name)
print(alice.introduce())

## 3. 实例属性与类属性

下面的`roles`写在类中，是类属性。两个实例会通过类访问同一个列表：

In [ ]:
class BadUser:
    roles: list[str] = []

    def __init__(self, name: str):
        self.name = name


first = BadUser("Alice")
second = BadUser("Bob")
first.roles.append("admin")

print(first.roles)
print(second.roles)
print(first.roles is second.roles)

如果每个实例应该拥有独立列表，就必须在`__init__()`中创建实例属性：

In [ ]:
class GoodUser:
    def __init__(self, name: str):
        self.name = name
        self.roles: list[str] = []


first = GoodUser("Alice")
second = GoodUser("Bob")
first.roles.append("admin")

print(first.roles)
print(second.roles)
print(first.roles is second.roles)

## 4. 实例方法

实例方法把与对象状态相关的操作放在类中：

```python
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self) -> None:
        self.value += 1
```

`None`表示这个方法负责修改对象状态，不需要返回业务结果。

## 5. 什么时候使用类

当一组数据和围绕这些数据的操作需要长期绑定在一起时，类通常很合适。例如：

- 聊天会话及其消息操作；
- 数据库连接及其查询操作；
- 模型客户端及其调用配置。

如果只是完成一个无状态计算，普通函数往往更简单。不要为了“面向对象”而把所有函数都包装成类。

## 6. 动手作业

打开同目录的`chat_session.py`，完成`ChatSession`的四个方法，然后保存并运行下面的测试。

特别注意：每个会话必须拥有独立的`messages`列表，不能把它写成共享的类属性。

In [ ]:
import importlib
import chat_session

importlib.reload(chat_session)
ChatSession = chat_session.ChatSession

first = ChatSession(1)
second = ChatSession(2)

first.add_message("user", "你好")
first.add_message("assistant", "你好！")

assert first.user_id == 1
assert first.message_count() == 2
assert second.message_count() == 0
assert first.messages is not second.messages
assert first.messages == [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！"},
]

first.clear()
assert first.message_count() == 0
print("第五课基础测试通过")

In [ ]:
def assert_raises(expected, function, *args):
    try:
        function(*args)
    except expected:
        return
    except Exception as error:
        raise AssertionError(
            f"期望{expected.__name__}，实际得到{type(error).__name__}"
        ) from error
    raise AssertionError(f"期望抛出{expected.__name__}，但没有异常")


assert_raises(TypeError, ChatSession, True)
session = ChatSession(1)
assert_raises(ValueError, session.add_message, "system", "提示")
assert_raises(ValueError, session.add_message, "user", "")
assert_raises(ValueError, session.add_message, "user", "   ")
print("第五课异常测试通过")

## 7. 理解题

1. 类和实例分别是什么？`ChatSession`与`ChatSession(1)`分别属于哪一种？
2. `self`表示什么？为什么调用`session.clear()`时不需要手动传入`self`？
3. 为什么`messages`必须在`__init__()`中创建，而不能直接写成类属性列表？
4. `clear()`为什么标注为`-> None`？